In [6]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import torch.nn as nn
from collections import defaultdict
from tqdm import tqdm
import os
from pathlib import Path

from sv3.nn import SvenWrapper, MLP
from sv3.sven import Sven
import matplotlib.pyplot as plt
plt.style.use("~/mpl_styles/cs_paper.mplstyle")

#device = torch.device('cuda') if torch.cuda.is_available() else (torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu'))
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

k_labels = {
    0.25: "B/4",
    0.5: "B/2",
    0.75: "3B/4",
    1.0: "B",
}
lr_labels = {
    1e-3:"10^{-3}",
    1e-4:"10^{-4}",
    1e-2:"10^{-2}",
}

outdir = "polynomial_plots/"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import pickle
import pandas as pd
# Load all experiment results and combine into single DataFrame
RESULTS_DIR = Path('../experiment_results/polynomial_scan/')

all_dfs = []
file_pattern = '*.pkl'

for filepath in sorted(RESULTS_DIR.glob(file_pattern)):
    with open(filepath, 'rb') as f:
        df_temp = pickle.load(f)
    all_dfs.append(df_temp)
    print(f"Loaded {filepath.name}: {len(df_temp)} rows")

scan_results = pd.concat(all_dfs, ignore_index=True)
batch_sizes = sorted(list(set(scan_results['batch_size'].values)))
k_fracs = sorted(list(set(scan_results['k_fraction'].values)))
lrs = sorted(list(set(scan_results[scan_results['optimizer']=='SVD']['lr'].values)))

Loaded polynomial_scan_results_dim6_deg4_bs128_width16_df.pkl: 24 rows
Loaded polynomial_scan_results_dim6_deg4_bs128_width32_df.pkl: 24 rows
Loaded polynomial_scan_results_dim6_deg4_bs128_width64_df.pkl: 24 rows
Loaded polynomial_scan_results_dim6_deg4_bs256_width16_df.pkl: 24 rows
Loaded polynomial_scan_results_dim6_deg4_bs256_width32_df.pkl: 24 rows
Loaded polynomial_scan_results_dim6_deg4_bs256_width64_df.pkl: 24 rows
Loaded polynomial_scan_results_dim6_deg4_bs32_width16_df.pkl: 24 rows
Loaded polynomial_scan_results_dim6_deg4_bs32_width32_df.pkl: 24 rows
Loaded polynomial_scan_results_dim6_deg4_bs32_width64_df.pkl: 24 rows
Loaded polynomial_scan_results_dim6_deg4_bs64_width16_df.pkl: 24 rows
Loaded polynomial_scan_results_dim6_deg4_bs64_width32_df.pkl: 24 rows
Loaded polynomial_scan_results_dim6_deg4_bs64_width64_df.pkl: 24 rows


In [9]:
LR_VAL = 0.1
K_FRAC = 0.5
for LR_VAL in [0.1,0.5,1.0]:
    for K_FRAC in [0.25,0.5,0.75,1.0]:
        plt.figure(figsize=(8,8))
        for bs in batch_sizes:
            k = (bs,K_FRAC,LR_VAL)
            cut = (scan_results['batch_size'] == bs) & (scan_results['k_fraction'] == K_FRAC) & (scan_results['lr'] == LR_VAL)
            res = scan_results[cut].iloc[0]
            n_epoch = len(res['losses']['train'])
            xvals = np.linspace(0,n_epoch,len(res['losses']['train_batch']))
            plt.plot(xvals, res['losses']['train_batch'], label=f'{bs}')
        plt.xlabel('Epoch')
        plt.ylabel('Training Loss',fontsize=16)
        plt.yscale('log')
        plt.ylim([1e-4,6])
        plt.legend(title='Batch Size')
        plt.title(rf"1D Regression, K$_\text{{SVD}} = {k_labels[K_FRAC]}, \; \eta = {LR_VAL}$")
        plt.tight_layout()
        OUT = f"{outdir}/trainLossByBatch_bsOverlay/"
        os.makedirs(OUT,exist_ok=True)
        plt.savefig(f"{OUT}/toy_1d_regression_trainLossByBatch_kfrac_{K_FRAC}_lr_{LR_VAL}_bsOverlay.pdf",bbox_inches='tight')
        plt.close()

In [10]:
LR_VAL = 0.1
K_FRAC = 0.5
for LR_VAL in [0.1,0.5,1.0]:
    for K_FRAC in [0.25,0.5,0.75,1.0]:
        plt.figure(figsize=(8,8))
        for bs in batch_sizes:
            k = (bs,K_FRAC,LR_VAL)
            cut = (scan_results['batch_size'] == bs) & (scan_results['k_fraction'] == K_FRAC) & (scan_results['lr'] == LR_VAL)
            res = scan_results[cut].iloc[0]
            n_epoch = len(res['losses']['train'])
            xvals = np.arange(1,n_epoch+1)
            plt.plot(xvals, res['losses']['train'], label=f'{bs}')
        plt.xlabel('Epoch')
        plt.ylabel('Training Loss',fontsize=16)
        plt.yscale('log')
        plt.ylim([1e-4,3])
        plt.legend(title='Batch Size')
        plt.title(rf"1D Regression, K$_\text{{SVD}} = {k_labels[K_FRAC]}, \; \eta = {LR_VAL}$")
        plt.tight_layout()
        OUT = f"{outdir}/trainLossByEpoch_bsOverlay/"
        os.makedirs(OUT,exist_ok=True)
        plt.savefig(f"{OUT}/toy_1d_regression_trainLossByEpoch_kfrac_{K_FRAC}_lr_{LR_VAL}_bsOverlay.pdf",bbox_inches='tight')
        plt.close()